In [1]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr
from anthropic import Anthropic 


In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


In [7]:
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("Missing ANTHROPIC_API_KEY in .env")

from anthropic import Anthropic
client = Anthropic(api_key=api_key)
MODEL = "claude-sonnet-4-20250514"

HF = os.getenv("HF_TOKEN")

In [8]:
folders = glob.glob("Information/*")
text_loader_kwargs = {'encoding': 'utf-8'}
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)


In [9]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

Created a chunk of size 1088, which is longer than the specified 1000


In [10]:
len(chunks)

123

In [11]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: products, company, contracts, employees


In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [24]:
# Load the existing vector store
db_name = "vector_db"
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

/tmp/ipykernel_9557/810779609.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)


In [25]:
print(f"Loaded existing vectorstore with {vectorstore._collection.count()} documents")

Loaded existing vectorstore with 123 documents


In [35]:
!pip uninstall -y langchain langchain-core langchain-community langchain-anthropic langchain-chroma langchain-text-splitters langsmith


Found existing installation: langchain 0.3.7
Uninstalling langchain-0.3.7:
  Successfully uninstalled langchain-0.3.7
Found existing installation: langchain-core 0.3.63
Uninstalling langchain-core-0.3.63:
  Successfully uninstalled langchain-core-0.3.63
Found existing installation: langchain-community 0.4
Uninstalling langchain-community-0.4:
  Successfully uninstalled langchain-community-0.4
Found existing installation: langchain-anthropic 1.0.0
Uninstalling langchain-anthropic-1.0.0:
  Successfully uninstalled langchain-anthropic-1.0.0
Found existing installation: langchain-chroma 1.0.0
Uninstalling langchain-chroma-1.0.0:
  Successfully uninstalled langchain-chroma-1.0.0
Found existing installation: langchain-text-splitters 0.3.8
Uninstalling langchain-text-splitters-0.3.8:
  Successfully uninstalled langchain-text-splitters-0.3.8
Found existing installation: langsmith 0.1.147
Uninstalling langsmith-0.1.147:
  Successfully uninstalled langsmith-0.1.147


In [26]:
from langchain_anthropic import ChatAnthropic
from langchain_community.vectorstores import Chroma
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain




# Initialize the Anthropic model (Claude)
llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.7)

# Memory to keep chat history
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True
)

# Use your existing vectorstore
retriever = vectorstore.as_retriever()

# Combine everything into a RAG conversation chain
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)


In [18]:
query = "Can you describe Insurellm in a few sentences"
result = conversation_chain.invoke({"question":query})
print(result["answer"])


Based on the context provided, I can see that Insurellm is a company that provides software solutions to insurance companies, but I don't have enough information in these contract excerpts to give you a comprehensive description of what Insurellm does as a company. 

The documents show that Insurellm offers products called "Homellm" and "Rellm" to insurance clients like Belvedere Insurance, Greenstone Insurance, and Pinnacle Insurance Co., and they provide technical support, training, and customer portals. However, the specific details about what these products do or what services Insurellm specializes in aren't described in the contract sections I have access to.

To give you a proper description of Insurellm, I would need more detailed information about their business model, products, and services.


In [27]:
def chat(message, history):
    result = conversation_chain.invoke({"question": message})
    return result["answer"]


In [28]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
